In [0]:
dbutils.widgets.text("event_blob_path", "")

event_blob_path = dbutils.widgets.get("event_blob_path")

if not event_blob_path:
    raise ValueError(
        "event_blob_path is empty. Run this notebook through ADF."
    )

print("Event blob path:", event_blob_path)

In [0]:
relative_path = event_blob_path.replace("bronze/", "", 1)

bronze_path = (
    "abfss://bronze@stcommercesrc.dfs.core.windows.net/"
    + relative_path
)

print("Bronze path:")
print(bronze_path)

In [0]:
print("STEP 1 - Reading Bronze order file")

df_bronze = spark.read.parquet(bronze_path)

rows_read = df_bronze.count()

print("Rows read from Bronze:", rows_read)

df_bronze.printSchema()

display(df_bronze.limit(10))

In [0]:
from pyspark.sql.functions import col, trim, upper

print("STEP 13.1 - Cleaning Payments")

df_clean = (
    df_bronze
    .withColumn("payment_method", upper(trim(col("payment_method"))))
    .withColumn("payment_status", upper(trim(col("payment_status"))))
    .withColumn("transaction_id", trim(col("transaction_id")))
    .withColumn("payment_amount", col("payment_amount").cast("decimal(12,2)"))
    .withColumn("refund_amount", col("refund_amount").cast("decimal(12,2)"))
)

print("Bronze rows:", df_bronze.count())
print("Rows after cleaning:", df_clean.count())

display(df_clean.limit(10))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

print("STEP 13.2 - Deduplicating Payments")

window_spec = (
    Window
    .partitionBy("payment_id")
    .orderBy(
        col("updated_at").desc_nulls_last(),
        col("created_at").desc_nulls_last()
    )
)

df_ranked = df_clean.withColumn(
    "row_num",
    row_number().over(window_spec)
)

df_deduplicated = (
    df_ranked
    .filter(col("row_num") == 1)
    .drop("row_num")
)

print("Rows before deduplication:", df_clean.count())
print("Rows after deduplication:", df_deduplicated.count())
print("Duplicates removed:", df_clean.count() - df_deduplicated.count())

In [0]:
print("STEP 13.3 - Validating Payments")

valid_payment_statuses = [
    "PENDING",
    "PROCESSING",
    "COMPLETED",
    "FAILED",
    "REFUNDED",
    "PARTIALLY_REFUNDED",
    "CANCELLED"
]

df_valid = df_deduplicated.filter(
    col("payment_id").isNotNull()
    & col("order_id").isNotNull()
    & col("payment_amount").isNotNull()
    & (col("payment_amount") > 0)
    & col("refund_amount").isNotNull()
    & (col("refund_amount") >= 0)
    & col("payment_status").isin(valid_payment_statuses)
)

df_rejected = df_deduplicated.join(
    df_valid.select("payment_id"),
    on="payment_id",
    how="left_anti"
)

print("Deduplicated rows:", df_deduplicated.count())
print("Valid Payments:", df_valid.count())
print("Rejected Payments:", df_rejected.count())

display(df_rejected.limit(10))

In [0]:
print("STEP 14.1 - Actual payment statuses")

display(
    df_deduplicated
    .groupBy("payment_status")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
print("STEP 14.2 - Payment validation diagnostics")

display(
    df_deduplicated.selectExpr(
        "COUNT(*) AS total_rows",
        "SUM(CASE WHEN payment_id IS NULL THEN 1 ELSE 0 END) AS null_payment_id",
        "SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id",
        "SUM(CASE WHEN payment_amount IS NULL THEN 1 ELSE 0 END) AS null_payment_amount",
        "SUM(CASE WHEN payment_amount <= 0 THEN 1 ELSE 0 END) AS nonpositive_payment_amount",
        "SUM(CASE WHEN refund_amount IS NULL THEN 1 ELSE 0 END) AS null_refund_amount",
        "SUM(CASE WHEN refund_amount < 0 THEN 1 ELSE 0 END) AS negative_refund_amount"
    )
)

In [0]:
valid_payment_statuses = [
    "PENDING",
    "PROCESSING",
    "COMPLETED",
    "FAILED",
    "REFUNDED",
    "PARTIALLY_REFUNDED",
    "CANCELLED"
]

display(
    df_deduplicated
    .filter(
        col("payment_status").isNull()
        | ~col("payment_status").isin(valid_payment_statuses)
    )
    .groupBy("payment_status")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
print("STEP 15.1 - Inspecting non-positive payment amounts")

display(
    df_deduplicated
    .filter(col("payment_amount") <= 0)
    .select(
        "payment_id",
        "order_id",
        "payment_amount",
        "payment_status",
        "refund_amount"
    )
    .limit(20)
)

In [0]:
print("STEP 15.2 - Corrected Payments validation")

valid_payment_statuses = [
    "SUCCESS",
    "FAILED",
    "PENDING",
    "REFUNDED"
]

df_valid = df_deduplicated.filter(
    col("payment_id").isNotNull()
    & col("order_id").isNotNull()
    & col("payment_amount").isNotNull()
    & (col("payment_amount") > 0)
    & col("refund_amount").isNotNull()
    & (col("refund_amount") >= 0)
    & col("payment_status").isin(valid_payment_statuses)
)

df_rejected = df_deduplicated.join(
    df_valid.select("payment_id"),
    on="payment_id",
    how="left_anti"
)

print("Deduplicated rows:", df_deduplicated.count())
print("Valid Payments:", df_valid.count())
print("Rejected Payments:", df_rejected.count())

In [0]:
print("STEP 16 - Payment amount investigation")

display(
    df_deduplicated
    .filter(col("payment_amount") <= 0)
    .groupBy("payment_amount", "payment_status")
    .count()
    .orderBy("payment_amount", "payment_status")
)

In [0]:
from pyspark.sql.functions import col

print("STEP 17 - Final Payments validation")

valid_payment_statuses = [
    "SUCCESS",
    "FAILED",
    "PENDING",
    "REFUNDED"
]

df_valid = df_deduplicated.filter(
    col("payment_id").isNotNull()
    & col("order_id").isNotNull()
    & col("payment_amount").isNotNull()
    & (col("payment_amount") >= 0)
    & col("refund_amount").isNotNull()
    & (col("refund_amount") >= 0)
    & col("payment_status").isin(valid_payment_statuses)
)

df_rejected = df_deduplicated.join(
    df_valid.select("payment_id"),
    on="payment_id",
    how="left_anti"
)

print("Deduplicated rows:", df_deduplicated.count())
print("Valid Payments:", df_valid.count())
print("Rejected Payments:", df_rejected.count())

In [0]:
from delta.tables import DeltaTable

print("STEP 18 - Merging Payments into Silver")

target_table = "ecommerce.silver.payments_external"

target = DeltaTable.forName(spark, target_table)

(
    target.alias("t")
    .merge(
        df_valid.alias("s"),
        "t.payment_id = s.payment_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Payments Silver MERGE completed successfully.")

In [0]:
%sql
SELECT * FROM ecommerce.silver.payments_external

payment_id,order_id,payment_method,transaction_id,payment_amount,payment_status,payment_date,refund_amount,created_at,updated_at
1,83811,UPI,TXN0000000001,36867.05,SUCCESS,2026-09-20T07:33:32.000Z,0.00,2026-09-20T07:33:32.000Z,2026-09-24T07:33:32.000Z
5,28658,CREDIT_CARD,TXN0000000005,53180.94,SUCCESS,2025-01-15T10:13:48.000Z,0.00,2025-01-15T10:13:48.000Z,2025-01-15T10:13:48.000Z
9,36422,CREDIT_CARD,TXN0000000009,86325.74,SUCCESS,2026-04-27T04:44:17.000Z,0.00,2026-04-27T04:44:17.000Z,2026-05-02T04:44:17.000Z
10,13397,UPI,TXN0000000010,68498.68,SUCCESS,2024-05-15T01:18:52.000Z,0.00,2024-05-15T01:18:52.000Z,2024-05-20T01:18:52.000Z
17,36435,NET_BANKING,TXN0000000017,75465.80,SUCCESS,2024-12-27T10:13:36.000Z,0.00,2024-12-27T10:13:36.000Z,2025-01-01T10:13:36.000Z
27,51857,COD,TXN0000000027,34937.68,SUCCESS,2026-07-04T09:25:01.000Z,0.00,2026-07-04T09:25:01.000Z,2026-07-08T09:25:01.000Z
28,18302,CREDIT_CARD,TXN0000000028,1873.89,SUCCESS,2025-03-08T03:42:04.000Z,0.00,2025-03-08T03:42:04.000Z,2025-03-16T03:42:04.000Z
33,55334,WALLET,TXN0000000033,14245.67,SUCCESS,2025-03-07T15:47:40.000Z,0.00,2025-03-07T15:47:40.000Z,2025-03-13T15:47:40.000Z
41,79819,CREDIT_CARD,TXN0000000041,40278.80,SUCCESS,2025-07-11T15:43:16.000Z,0.00,2025-07-11T15:43:16.000Z,2025-07-13T15:43:16.000Z
44,40307,CREDIT_CARD,TXN0000000044,63970.27,SUCCESS,2025-02-14T22:03:09.000Z,0.00,2025-02-14T22:03:09.000Z,2025-02-23T22:03:09.000Z
